In [1]:
!pip -q install opencv-python-headless scikit-image timm


   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 363.4/363.4 MB 4.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 13.8/13.8 MB 98.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 24.6/24.6 MB 78.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 883.7/883.7 kB 46.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 664.8/664.8 MB 2.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 211.5/211.5 MB 8.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 56.3/56.3 MB 14.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 127.9/127.9 MB 13.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 207.5/207.5 MB 8.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 21.1/21.1 MB 79.2 MB/s eta 0:00:00


In [2]:
%%writefile p3_stage2_cmf.py
import os, math
from typing import Tuple
import torch
import torch.nn as nn
import torch.nn.functional as F
from torchvision.models import resnet50

# ---------- helpers ----------
class GlobalAvgPool(nn.Module):
    def forward(self, x):  # (B,C,H,W)->(B,C)
        return x.mean(dim=(2,3))

# ---------- FFT adapter (magnitude only for Stage-2) ----------
class FFTMagAdapter(nn.Module):
    def __init__(self, in_ch: int, out_ch: int):
        super().__init__()
        self.reduce = nn.Conv2d(in_ch, out_ch, kernel_size=1, bias=False)
        self.bn = nn.BatchNorm2d(out_ch)
        self.act = nn.ReLU(inplace=True)

    def forward(self, x):  # x: (B,C,H,W)
        X = torch.fft.fft2(x, dim=(-2,-1))
        mag = torch.abs(X)  # magnitude
        mag = torch.log1p(mag)
        # per-sample standardize
        mean = mag.mean(dim=(2,3), keepdim=True)
        std  = mag.std(dim=(2,3), keepdim=True) + 1e-6
        mag = (mag - mean) / std
        f = self.reduce(mag)
        f = self.bn(f)
        return self.act(f)  # (B,out_ch,H,W)

# ---------- Cross-Modal Fusion (Q from RGB, K/V from Freq) ----------
class CMF(nn.Module):
    """
    Cross-modal attention on spatial maps:
      Q from RGB map (B,C,H,W)
      K,V from Freq map (B,C,H,W)
    We flatten spatial dims -> tokens, do MH-Attn in low dim (d_model=128), then reshape back.
    """
    def __init__(self, in_ch: int, d_model: int = 128, num_heads: int = 2, dropout: float = 0.0):
        super().__init__()
        self.rgb_proj = nn.Conv2d(in_ch, d_model, 1, bias=False)
        self.freq_proj = nn.Conv2d(in_ch, d_model, 1, bias=False)
        encoder_layer = nn.TransformerEncoderLayer(
            d_model=d_model, nhead=num_heads, dim_feedforward=d_model*2,
            dropout=dropout, batch_first=True, activation="gelu"
        )
        # Use cross-attention by concatenating K/V source; here we emulate cross-attn via one encoder layer on [Q;context]
        # Simpler: use MultiheadAttention directly
        self.attn = nn.MultiheadAttention(d_model, num_heads, batch_first=True, dropout=dropout)
        self.out_proj = nn.Conv2d(d_model, in_ch, 1, bias=False)
        self.out_bn = nn.BatchNorm2d(in_ch)

    def forward(self, rgb_map, freq_map):  # both (B,C,H,W)
        B,C,H,W = rgb_map.shape
        q = self.rgb_proj(rgb_map).flatten(2).transpose(1,2)   # (B,HW,d)
        k = self.freq_proj(freq_map).flatten(2).transpose(1,2) # (B,HW,d)
        v = k
        fused, _ = self.attn(q, k, v)                          # (B,HW,d)
        fused = fused.transpose(1,2).reshape(B, -1, H, W)      # (B,d,H,W)
        fused = self.out_proj(fused)
        fused = self.out_bn(fused)
        return fused  # (B,C,H,W)

# ---------- Tiny encoder (kept frozen in Stage-2) ----------
class TinyTransformerEncoder(nn.Module):
    def __init__(self, dim=1024, depth=2, heads=8, mlp_ratio=2.0, dropout=0.0):
        super().__init__()
        layer = nn.TransformerEncoderLayer(
            d_model=dim, nhead=heads, dim_feedforward=int(dim*mlp_ratio),
            dropout=dropout, batch_first=True, activation='gelu'
        )
        self.enc = nn.TransformerEncoder(layer, num_layers=depth)

    def forward(self, x, src_key_padding_mask=None):  # (B,T,dim)
        return self.enc(x, src_key_padding_mask=src_key_padding_mask)

# ---------- Superpixel token pooling ----------
import numpy as np
from skimage.segmentation import slic
from torchvision import transforms

class SPTokenPool(nn.Module):
    """
    Pools (B,C,H,W) into variable #region tokens using SLIC masks.
    """
    def forward(self, feat_map, seg):  # seg: (B,H0,W0) ints
        B,C,H,W = feat_map.shape
        seg = F.interpolate(seg.unsqueeze(1).float(), size=(H,W), mode='nearest').squeeze(1).long()  # (B,H,W)
        region_tokens, counts = [], []
        for b in range(B):
            ids = torch.unique(seg[b])
            feats = feat_map[b]  # (C,H,W)
            toks = []
            for rid in ids:
                m = (seg[b]==rid).float()              # (H,W)
                w = m / (m.sum()+1e-6)
                tok = (feats * w).sum(dim=(1,2))       # (C)
                toks.append(tok)
            toks = torch.stack(toks, 0)                 # (R,C)
            region_tokens.append(toks); counts.append(toks.size(0))
        maxR = max(counts)
        padded = []
        for t in region_tokens:
            if t.size(0) < maxR:
                pad = torch.zeros(maxR - t.size(0), t.size(1), device=t.device, dtype=t.dtype)
                t = torch.cat([t, pad], 0)
            padded.append(t)
        toks = torch.stack(padded, 0)                   # (B,R,C)
        mask = torch.arange(maxR, device=toks.device)[None,:] >= torch.tensor(counts, device=toks.device)[:,None]
        return toks, mask  # mask True=pad

# ---------- Full Stage-2 model ----------
class P3Stage2CMF(nn.Module):
    def __init__(self, n_segments=50, freeze_backbone=True, freeze_encoder=True, d_model=128, heads=2):
        super().__init__()
        # Backbone
        self.backbone = resnet50(weights=None)
        self.backbone.fc = nn.Identity()
        # Neck to 1024
        self.neck = nn.Conv2d(2048, 1024, 1, bias=False)
        self.neck_bn = nn.BatchNorm2d(1024); self.neck_act = nn.ReLU(inplace=True)

        # Frequency adapter and CMF
        self.fft = FFTMagAdapter(1024, 1024)
        self.cmf = CMF(1024, d_model=d_model, num_heads=heads)

        # Tokenization (SLIC)
        self.pool_sp = SPTokenPool()
        self.encoder = TinyTransformerEncoder(dim=1024, depth=2, heads=8, mlp_ratio=2.0)

        # Head
        self.head = nn.Sequential(nn.Linear(1024, 512), nn.ReLU(inplace=True), nn.Linear(512, 2))
        self.gap = GlobalAvgPool()

        # Freeze knobs
        if freeze_backbone:
            for p in self.backbone.parameters(): p.requires_grad = False
            for p in self.neck.parameters(): p.requires_grad = False
            for p in self.neck_bn.parameters(): p.requires_grad = False
        if freeze_encoder:
            for p in self.encoder.parameters(): p.requires_grad = False

    def forward(self, x, seg):
        # backbone
        x = self.backbone.conv1(x); x = self.backbone.bn1(x); x = self.backbone.relu(x); x = self.backbone.maxpool(x)
        x = self.backbone.layer1(x); x = self.backbone.layer2(x); x = self.backbone.layer3(x); x = self.backbone.layer4(x)
        h = self.neck_act(self.neck_bn(self.neck(x)))  # (B,1024,h,w)

        # frequency + CMF (residual into RGB)
        f = self.fft(h.detach())  # adapter learns; keep backbone frozen
        fused = h + self.cmf(h, f)  # residual add

        # superpixel tokens
        toks, pad_mask = self.pool_sp(fused, seg)  # (B,R,1024), (B,R)
        enc = self.encoder(toks, src_key_padding_mask=pad_mask)  # (B,R,1024)
        enc = enc.mean(dim=1)  # simple pooling over regions
        logits = self.head(enc)  # (B,2)
        return logits


Writing p3_stage2_cmf.py


In [3]:
%%writefile run_stage2.py
import argparse, os, torch, torch.nn as nn, torch.optim as optim
from torch.utils.data import DataLoader
from torchvision import datasets, transforms
from torchvision.models import resnet50, ResNet50_Weights
import numpy as np
from skimage.segmentation import slic

from p3_stage2_cmf import P3Stage2CMF

from sklearn.metrics import roc_auc_score, f1_score, accuracy_score, roc_curve
from scipy.optimize import brentq
from scipy.interpolate import interp1d

def calc_metrics(y_true, y_prob):
    y_pred = (y_prob > 0.5).astype(int)
    auc  = roc_auc_score(y_true, y_prob)
    f1   = f1_score(y_true, y_pred)
    acc  = accuracy_score(y_true, y_pred)
    fpr, tpr, _ = roc_curve(y_true, y_prob)
    eer  = brentq(lambda x: 1. - x - interp1d(fpr, tpr)(x), 0., 1.)
    return auc, f1, eer, acc


# ----- superpixel dataset -----
class SuperpixelImageFolder(datasets.ImageFolder):
    def __init__(self, root, transform=None, n_segments=50, compactness=10.0, size=320):
        super().__init__(root, transform=transform)
        self.n_segments = n_segments
        self.compactness = compactness
        self.size = size
        self.denorm_mean = torch.tensor([0.485,0.456,0.406])[:,None,None]
        self.denorm_std  = torch.tensor([0.229,0.224,0.225])[:,None,None]

    def __getitem__(self, i):
        x, y = super().__getitem__(i)  # x: [3,H,W] normalized
        x_vis = (x*self.denorm_std + self.denorm_mean).clamp(0,1).permute(1,2,0).cpu().numpy()
        seg = slic(x_vis, n_segments=self.n_segments, compactness=self.compactness, start_label=0).astype(np.int32)
        seg = torch.from_numpy(seg)  # (H,W)
        return x, seg, y

def sp_collate(batch):
    xs, segs, ys = zip(*batch)
    xs = torch.stack(xs, 0)
    segs = torch.stack(segs, 0)
    ys = torch.tensor(ys, dtype=torch.long)
    return xs, segs, ys

def make_loaders(data_root: str, batch_size: int = 16, size=320, n_segments=50):
    tf_train = transforms.Compose([
        transforms.Resize((size,size)),
        transforms.RandomHorizontalFlip(),
        transforms.ToTensor(),
        transforms.Normalize([0.485,0.456,0.406],[0.229,0.224,0.225])
    ])
    tf_eval = transforms.Compose([
        transforms.Resize((size,size)),
        transforms.ToTensor(),
        transforms.Normalize([0.485,0.456,0.406],[0.229,0.224,0.225])
    ])
    tr = SuperpixelImageFolder(os.path.join(data_root,"train"), transform=tf_train, n_segments=n_segments)
    va = SuperpixelImageFolder(os.path.join(data_root,"valid"), transform=tf_eval,  n_segments=n_segments)
    te = SuperpixelImageFolder(os.path.join(data_root,"test"),  transform=tf_eval,  n_segments=n_segments)
    dl_tr = DataLoader(tr, batch_size=batch_size, shuffle=True,  num_workers=2, pin_memory=True, collate_fn=sp_collate)
    dl_va = DataLoader(va, batch_size=batch_size, shuffle=False, num_workers=2, pin_memory=True, collate_fn=sp_collate)
    dl_te = DataLoader(te, batch_size=batch_size, shuffle=False, num_workers=2, pin_memory=True, collate_fn=sp_collate)
    return dl_tr, dl_va, dl_te

# ----- training utils -----
@torch.no_grad()
def accuracy(logits, y):
    return (logits.argmax(1)==y).float().mean().item()

def train_one_epoch(model, dl, opt, device):
    model.train()
    ce = nn.CrossEntropyLoss()
    totL=totA=n=0
    for x,seg,y in dl:
        x,seg,y = x.to(device), seg.to(device), y.to(device)
        opt.zero_grad(set_to_none=True)
        logits = model(x, seg)
        loss = ce(logits, y)
        loss.backward(); opt.step()
        b = x.size(0); totL += loss.item()*b; totA += accuracy(logits.detach(), y)*b; n += b
    return totL/n, totA/n

@torch.no_grad()
def evaluate(model, dl, device):
    model.eval()
    ce = nn.CrossEntropyLoss()
    totL=totA=n=0
    for x,seg,y in dl:
        x,seg,y = x.to(device), seg.to(device), y.to(device)
        logits = model(x, seg)
        loss = ce(logits, y)
        b = x.size(0); totL += loss.item()*b; totA += accuracy(logits, y)*b; n += b
    return totL/n, totA/n

def load_shape_compatible(m, path):
    ckpt = torch.load(path, map_location='cpu')
    if isinstance(ckpt, dict) and "model" in ckpt: ckpt = ckpt["model"]
    msd = m.state_dict()
    keep = {k:v for k,v in ckpt.items() if k in msd and v.shape == msd[k].shape}
    m.load_state_dict(keep, strict=False)
    print(f"Loaded {len(keep)} tensors; skipped {len(list(set(ckpt.keys())-set(keep.keys())))} (shape mismatch).")

def main():
    p = argparse.ArgumentParser("Stage-2 CMF (RGB↔Freq) with superpixels; freeze big parts")
    p.add_argument('--data_root', type=str, required=True)
    p.add_argument('--batch_size', type=int, default=16)
    p.add_argument('--epochs', type=int, default=3)
    p.add_argument('--lr', type=float, default=5e-4)
    p.add_argument('--n_segments', type=int, default=50)
    p.add_argument('--freeze_backbone', action='store_true')
    p.add_argument('--freeze_encoder', action='store_true')
    p.add_argument('--imagenet_backbone', action='store_true')
    p.add_argument('--ckpt', type=str, default='', help='(optional) Stage-1 checkpoint to warm start')
    p.add_argument('--save', type=str, default='/kaggle/working/p3_stage2_cmf.pth')
    args = p.parse_args()

    device = 'cuda' if torch.cuda.is_available() else 'cpu'
    tr, va, te = make_loaders(args.data_root, batch_size=args.batch_size, n_segments=args.n_segments)

    model = P3Stage2CMF(n_segments=args.n_segments,
                        freeze_backbone=args.freeze_backbone,
                        freeze_encoder=args.freeze_encoder).to(device)

    # optional: load ImageNet weights into backbone
    if args.imagenet_backbone:
        try:
            print("Loading ImageNet weights into ResNet…")
            model.backbone.load_state_dict(resnet50(weights=ResNet50_Weights.IMAGENET1K_V1).state_dict(), strict=False)
            print("✓ Loaded.")
        except Exception as e:
            print(f"Could not load ImageNet weights: {e}")

    # warm start from Stage-1 FFT checkpoint if provided (loads matching layers only)
    if args.ckpt and os.path.isfile(args.ckpt):
        print(f"Warm start from {args.ckpt}")
        load_shape_compatible(model, args.ckpt)

    opt = optim.AdamW([p for p in model.parameters() if p.requires_grad], lr=args.lr, weight_decay=1e-4)

    best = 0.0
    for ep in range(1, args.epochs+1):
        trL,trA = train_one_epoch(model, tr, opt, device)
        
             # ---- DeepfakeBench metrics on validation ----
        model.eval(); y_true, y_prob = [], []
        with torch.no_grad():
            for x, seg, y in va:
                x, seg, y = x.to(device), seg.to(device), y.to(device)
                probs = torch.softmax(model(x, seg), dim=1)[:, 1].cpu().numpy()
                y_true.extend(y.cpu().numpy())
                y_prob.extend(probs)
        
        auc, f1, eer, acc = calc_metrics(np.array(y_true), np.array(y_prob))
        print(f"Epoch {ep:02d} | train {trL:.4f}/{trA:.4f} | "
              f"valid AUROC={auc:.3f} | F1={f1:.3f} | EER={eer:.3f} | ACC={acc:.3f}")
        
        # Save checkpoint by best AUROC (DeepfakeBench standard)
        if auc > best:
            best = auc
            torch.save({"model": model.state_dict()}, args.save)
            print(f"  ↳ [BEST] Saved model with AUC={best:.4f} to {args.save}")


    teL,teA = evaluate(model, te, device)
    print(f"Test: loss {teL:.4f} acc {teA:.4f}")

if __name__ == "__main__":
    main()


Writing run_stage2.py


In [4]:
DATA_ROOT = "/kaggle/input/140k-real-and-fake-faces/real_vs_fake/real-vs-fake"
!ls -1 $DATA_ROOT; ls -1 $DATA_ROOT/train; ls -1 $DATA_ROOT/valid; ls -1 $DATA_ROOT/test


test
train
valid
fake
real
fake
real
fake
real


In [5]:

!python run_stage2.py \
  --data_root $DATA_ROOT \
  --batch_size 16 --epochs 3 --lr 5e-4 \
  --n_segments 50 \
  --freeze_backbone --freeze_encoder \
  --imagenet_backbone \
  --save /kaggle/working/p3_stage2_cmf.pth


Loading ImageNet weights into ResNet…
Downloading: "https://download.pytorch.org/models/resnet50-0676ba61.pth" to /root/.cache/torch/hub/checkpoints/resnet50-0676ba61.pth
100%|███████████████████████████████████████| 97.8M/97.8M [00:00<00:00, 223MB/s]
✓ Loaded.
/usr/local/lib/python3.11/dist-packages/torch/nn/modules/transformer.py:508: UserWarning: The PyTorch API of nested tensors is in prototype stage and will change in the near future. We recommend specifying layout=torch.jagged when constructing a nested tensor, as this layout receives active development, has better operator coverage, and works with torch.compile. (Triggered internally at /pytorch/aten/src/ATen/NestedTensorImpl.cpp:178.)
  output = torch._nested_tensor_from_mask(
Epoch 01 | train 0.3558/0.8447 | valid AUROC=0.958 | F1=0.866 | EER=0.109 | ACC=0.875
  ↳ [BEST] Saved model with AUC=0.9578 to /kaggle/working/p3_stage2_cmf.pth
Epoch 02 | train 0.3079/0.8702 | valid AUROC=0.964 | F1=0.874 | EER=0.103 | ACC=0.883
  ↳ [BE

In [6]:
!python -u run_stage2.py \
  --data_root $DATA_ROOT \
  --epochs 0 \
  --n_segments 50 \
  --freeze_backbone --freeze_encoder \
  --ckpt /kaggle/working/p3_stage2_cmf.pth


Warm start from /kaggle/working/p3_stage2_cmf.pth
Loaded 370 tensors; skipped 0 (shape mismatch).
/usr/local/lib/python3.11/dist-packages/torch/nn/modules/transformer.py:508: UserWarning: The PyTorch API of nested tensors is in prototype stage and will change in the near future. We recommend specifying layout=torch.jagged when constructing a nested tensor, as this layout receives active development, has better operator coverage, and works with torch.compile. (Triggered internally at /pytorch/aten/src/ATen/NestedTensorImpl.cpp:178.)
  output = torch._nested_tensor_from_mask(
Test: loss 0.2139 acc 0.9156


In [7]:
from run_stage2 import make_loaders
dl_tr, dl_va, dl_te = make_loaders(DATA_ROOT, batch_size=2, n_segments=50)
x, seg, y = next(iter(dl_tr))
print("img:", x.shape, "seg:", seg.shape, "unique regions in sample0:", int(seg[0].unique().numel()))


img: torch.Size([2, 3, 320, 320]) seg: torch.Size([2, 320, 320]) unique regions in sample0: 39


In [8]:
!ls -lh /kaggle/working/p3_stage2_cmf.pth


-rw-r--r-- 1 root root 170M Nov  2 11:38 /kaggle/working/p3_stage2_cmf.pth
